# 04 - Particionamiento y Adaptive Query Execution (AQE)

### Diferencia técnica entre operaciones
* **`repartition(n)`**: Realiza un shuffle completo en la red. Útil para balancear particiones sesgadas.
* **`coalesce(n)`**: Reduce particiones combinando bloques locales sin generar shuffle.


In [ ]:
import sys
sys.path.append("..")
from src.config import get_spark_session
import pyspark.sql.functions as F

spark = get_spark_session("04_AQE")
df_num = spark.range(0, 500000, 1, numPartitions=8)
print("Particiones iniciales :", df_num.rdd.getNumPartitions())
print("Tras coalesce(2)      :", df_num.coalesce(2).rdd.getNumPartitions())
print("Tras repartition(10)  :", df_num.repartition(10).rdd.getNumPartitions())


### Ajuste Dinámico con AQE
AQE coalescee particiones intermedias automáticamente al detectar etapas con volúmenes bajos:


In [ ]:
df_agg = df_num.groupBy(F.col("id") % 4).count()
df_agg.explain(mode="cost")
df_agg.show()
